Fit the damping ratio from the decay of the simulated peaks.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np

t, x, v = np.loadtxt("data/sim.csv", delimiter=",", skiprows=1).T
with open("results/sim.json") as f:
    sim = json.load(f)
# Peaks decay as exp(-zeta omega t)
is_peak = (x[1:-1] > x[:-2]) & (x[1:-1] > x[2:])
t_peaks, x_peaks = t[1:-1][is_peak], x[1:-1][is_peak]
slope = np.polyfit(t_peaks, np.log(x_peaks), 1)[0]
zeta = -slope / sim["omega"]
os.makedirs("results", exist_ok=True)
with open("results/fit.json", "w") as f:
    json.dump({"zeta": zeta, "n_peaks": int(is_peak.sum())}, f, indent=2)

In [ ]:
# The peaks and the fit through their logarithm, for the report to show
np.savetxt(
    "results/peaks.csv",
    np.column_stack([t_peaks, x_peaks]),
    delimiter=",",
    header="t,x",
    comments="",
    fmt="%.4f",
)
os.makedirs("figures", exist_ok=True)
# Fixed ids in the SVG, so the same plot is the same file
plt.rcParams["svg.hashsalt"] = "peaks"
fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(t_peaks, x_peaks, "o", label="Peaks")
fit = np.polyfit(t_peaks, np.log(x_peaks), 1)
ax.semilogy(t_peaks, np.exp(np.polyval(fit, t_peaks)), "--", label="Fit")
ax.set_xlabel("t")
ax.set_ylabel("Peak x")
ax.legend()
fig.savefig("figures/peaks.svg", metadata={"Date": None})
plt.close(fig)

The fit gives ζ = `{python} f"{zeta:.4f}"`, against `{python} sim["zeta"]` in the simulation.

In [ ]:
#| label: fig-decay
#| fig-cap: Displacement and fitted envelope.
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, x, label="Simulated")
ax.plot(t, np.exp(-zeta * sim["omega"] * t), "--", label="Fitted envelope")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.legend();